In [1]:
import anndata as ad

adata=ad.read_h5ad("../data/raw/sciplex3.h5ad", backed="r")
print(adata)

AnnData object with n_obs × n_vars = 581777 × 58347 backed at '../data/raw/sciplex3.h5ad'
    obs: 'cell_type', 'dose', 'dose_character', 'dose_pattern', 'g1s_score', 'g2m_score', 'pathway', 'pathway_level_1', 'pathway_level_2', 'product_dose', 'product_name', 'proliferation_index', 'replicate', 'size_factor', 'target', 'vehicle'
    var: 'id', 'num_cells_expressed-0-0', 'num_cells_expressed-1-0', 'num_cells_expressed-1'


In [2]:
print(adata.obs.head(1))
print(adata.obs.columns.tolist())

                                 cell_type    dose dose_character  \
index                                                               
A01_E09_RT_BC_100_Lig_BC_245-0-0      A549  1000.0           1000   

                                 dose_pattern  g1s_score  g2m_score  \
index                                                                 
A01_E09_RT_BC_100_Lig_BC_245-0-0            2   1.155964   2.475312   

                                        pathway pathway_level_1  \
index                                                             
A01_E09_RT_BC_100_Lig_BC_245-0-0  TGF-beta/Smad   PKC signaling   

                                 pathway_level_2                 product_dose  \
index                                                                           
A01_E09_RT_BC_100_Lig_BC_245-0-0   PKC activitiy  Enzastaurin (LY317615)_1000   

                                            product_name  proliferation_index  \
index                                              

In [3]:
for col in adata.obs.columns:
    print(col, adata.obs[col].nunique())

cell_type 3
dose 5
dose_character 5
dose_pattern 4
g1s_score 150483
g2m_score 177064
pathway 22
pathway_level_1 17
pathway_level_2 55
product_dose 753
product_name 189
proliferation_index 204478
replicate 2
size_factor 31035
target 88
vehicle 2


In [4]:
obs=adata.obs
print(obs["cell_type"].value_counts())

cell_type
MCF7    292010
K562    146752
A549    143015
Name: count, dtype: int64


In [9]:
a549 = obs[obs["cell_type"] == "A549"]
counts = a549["product_dose"].value_counts()
print(counts.describe())

count     753.000000
mean      189.926959
std       123.231862
min        20.000000
25%       154.000000
50%       198.000000
75%       220.000000
max      3287.000000
Name: count, dtype: float64


In [10]:
print("A549 vehicle cells:", a549["vehicle"].sum())

A549 vehicle cells: 3287


In [22]:
print(sorted(obs["dose"].unique()))

[np.float64(0.0), np.float64(10.0), np.float64(100.0), np.float64(1000.0), np.float64(10000.0)]


In [27]:
a549["dose"].value_counts().sort_index()

dose
0.0         3287
10.0       37414
100.0      35676
1000.0     35138
10000.0    31500
Name: count, dtype: int64

In [28]:
import pandas as pd
print(pd.crosstab(a549["vehicle"], a549["dose"] == 0))
print(a549.loc[a549["dose"] == 0, "product_name"].unique())

dose      False  True 
vehicle               
0        139728      0
1             0   3287
['Vehicle']
Categories (189, object): ['2-Methoxyestradiol (2-MeOE2)', '(+)-JQ1', 'A-366', 'ABT-737', ..., 'XAV-939', 'YM155 (Sepantronium Bromide)', 'ZM 447439', 'Zileuton']


In [30]:
adata.var

,id,num_cells_expressed-0-0,num_cells_expressed-1-0,num_cells_expressed-1
index,,,,
TSPAN6,ENSG00000000003.14,4017,13320,199
TNMD,ENSG00000000005.5,3,14,3
DPM1,ENSG00000000419.12,8442,72353,11231
SCYL3,ENSG00000000457.13,4386,24120,3380
C1orf112,ENSG00000000460.16,5998,24454,9555
...,...,...,...,...
AL591163.1,ENSG00000284744.1,8,26,27
AL589702.1,ENSG00000284745.1,2,3,0
AC068587.10,ENSG00000284746.1,0,0,0


In [31]:
import numpy as np
mask=(adata.obs["cell_type"]=="A549").values
a549=adata[mask].to_memory()

In [33]:
for col in a549.obs.select_dtypes("category").columns:
    a549.obs[col] = a549.obs[col].cat.remove_unused_categories()

In [ ]:
print(a549)

AnnData object with n_obs × n_vars = 143015 × 58347
    obs: 'cell_type', 'dose', 'dose_character', 'dose_pattern', 'g1s_score', 'g2m_score', 'pathway', 'pathway_level_1', 'pathway_level_2', 'product_dose', 'product_name', 'proliferation_index', 'replicate', 'size_factor', 'target', 'vehicle'
    var: 'id', 'num_cells_expressed-0-0', 'num_cells_expressed-1-0', 'num_cells_expressed-1'


In [40]:
a549.write_h5ad("../data/processed/a549_raw.h5ad", compression="gzip")